In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "clean").exists() else Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

# 1. 기업개요_최종: 회사 마스터 테이블 (crno가 유니크 키)
corp = pd.read_csv(CLEAN_DIR / "기업개요_최종.csv", dtype=str, encoding="utf-8-sig")
corp = corp.drop(columns=["Unnamed: 0"])
master = corp.set_index("crno")

MASTER_COLS = ["corpNm", "corpEnsnNm", "enpBsadr", "enpEstbDt", "상장여부"]

def attach_master(df, id_col, prefix):
    sub = master[MASTER_COLS].add_prefix(f"{prefix}_")
    return df.merge(sub, left_on=id_col, right_index=True, how="left")

# 2. 계열회사_전처리: crno(모기업) - afilCmpyCrno(계열사), 양쪽 다 ID 보유 -> 양쪽 다 ID 매칭
afil = pd.read_csv(CLEAN_DIR / "계열회사_전처리.csv", dtype=str, encoding="utf-8-sig")
afil["relation_type"] = "계열회사"
afil = afil.rename(columns={"crno": "source_crno", "afilCmpyCrno": "target_crno", "afilCmpyNm": "target_name_raw"})
afil = attach_master(afil, "source_crno", "source")
afil = attach_master(afil, "target_crno", "target")

# 3. 종속기업_정리: crno(모기업)만 ID 보유, 종속기업 자체 ID는 없음 -> 모기업 쪽만 ID 매칭
sub = pd.read_csv(CLEAN_DIR / "종속기업_정리.csv", dtype=str, encoding="utf-8-sig")
sub["relation_type"] = "종속기업"
sub = sub.rename(columns={"crno": "source_crno", "sbrdEnpNm": "target_name_raw"})
sub["target_crno"] = pd.NA
sub = attach_master(sub, "source_crno", "source")

# 4. 공통 스키마로 통합 (세로 결합)
common_cols = [
    "relation_type", "source_crno", "source_corpNm", "source_corpEnsnNm", "source_enpBsadr", "source_enpEstbDt", "source_상장여부",
    "target_crno", "target_name_raw", "target_corpNm", "target_corpEnsnNm", "target_enpBsadr", "target_enpEstbDt", "target_상장여부",
]

afil_out = afil.reindex(columns=common_cols)
sub_out = sub.reindex(columns=common_cols)

# 종속기업 전용 부가 속성
sub_out["sbrdEnpMainBizCtt"] = sub["sbrdEnpMainBizCtt"].values
sub_out["domestic"] = sub["domestic"].values
sub_out["name_norm"] = sub["name_norm"].values

merged = pd.concat([afil_out, sub_out], ignore_index=True, sort=False)

merged.to_csv(CLEAN_DIR / "지식그래프_통합_ID매칭.csv", index=False, encoding="utf-8-sig")